# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [9]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI


In [10]:
# Initialize and constants

"""
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
"""

OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')


In [11]:
links = fetch_website_links("https://edwarddonner.com")
links


['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https:/

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [12]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""


In [13]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt


In [ ]:
print(get_links_user_prompt("https://edwarddonner.com"))


In [14]:
def select_relevant_links(url):
    MODEL = 'qwen2.5:3b'
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = ollama.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    print(result)
    # Clean up common markdown formatting if the model included it
    result = result.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    print(result)
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links
    

In [ ]:
select_relevant_links("https://edwarddonner.com")


In [15]:
def select_relevant_links(url):    
    MODEL = 'qwen2.5:3b'
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = ollama.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content

    # Clean up common markdown formatting if the model included it
    result = result.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    print(result)
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links


In [16]:
select_relevant_links("https://edwarddonner.com")


Selecting relevant links for https://edwarddonner.com by calling qwen2.5:3b
{
    "links": [
        {"type": "about page", "url": "https://edwarddonner.com/about-me-and-about-nebula/"},
        {"type": "company page", "url": "https://edwarddonner.com/"},
        {"type": "careers page", "url": "https://nebula.io/?utm_source=ed&utm_medium=referral"}
    ]
}
Found 3 relevant links


{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'company page', 'url': 'https://edwarddonner.com/'},
  {'type': 'careers page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'}]}

In [ ]:
select_relevant_links("https://huggingface.co")


## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [17]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result


In [ ]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))


In [18]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [19]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt


In [20]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")


Selecting relevant links for https://huggingface.co by calling qwen2.5:3b
{
    "links": [
        {"type": "about page", "url": "https://huggingface.co/about"},
        {"type": "careers page", "url": "https://huggingface.co/join"},
        {"type": "enterprise page", "url": "https://huggingface.co/enterprise"},
        {"type": "pricing page", "url": "https://huggingface.co/pricing"}
    ]
}
Found 4 relevant links


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


"\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nHardware\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nNEW\nMicroduck: A Tiny Robot for AI Builders 🦆\nGoogle Gemma 4 is here 💫\nStorage Buckets: AI-native object storage\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\

In [21]:
def create_brochure(company_name, url):
    MODEL = 'qwen2.5:3b'
    print(f"Creating brochure for {company_name} by calling {MODEL}")
    response = ollama.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))


In [22]:
create_brochure("HuggingFace", "https://huggingface.co")


Creating brochure for HuggingFace by calling qwen2.5:3b
Selecting relevant links for https://huggingface.co by calling qwen2.5:3b
{
    "links": [
        {"type": "about page", "url": "https://huggingface.co/"},
        {"type": "careers page", "url": "https://huggingface.co/join"},
        {"type": "enterprise page", "url": "https://huggingface.co/enterprise"},
        {"type": "pricing page", "url": "https://huggingface.co/pricing"},
        {"type": "learn page", "url": "https://huggingface.co/learn"}
    ]
}
Found 5 relevant links


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


# Hugging Face: Building the Future of Machine Learning
## Welcome to Hugging Face
The AI community building the future 🌟
Hugging Face is a platform that serves as the heart of a vibrant community of AI engineers and researchers. It serves as the collaborative hub where members of this community come together to develop, discover, and explore one-of-a-kind AI models and datasets. Hugging Face fosters innovation and knowledge sharing within its tight-knit group of global AI builders.

### Company Culture
At Hugging Face, we celebrate the work of machine learning and AI developers. We embrace openness, collaboration and growth. Our culture encourages our employees to strive for excellence and pushes them to reach new heights. At the core of our culture is our commitment to fostering a diversity of perspectives, beliefs, and viewpoints and to being a catalyst for open-mindedness and constructive discussions.

### Customers
Hugging Face appeals to many different types of users. For users, you can find a wide spectrum of applications that can be used directly with Hugging Face’s models and datasets. For companies, Hugging Face offers cutting-edge solutions and tools to support their AI initiatives. Whether you are a tech company, a software developer, or just a curious AI enthusiast – we got you covered at Hugging Face.

### Careers At Hugging Face
Hugging Face values diversity and inclusivity. As a global platform for the machine learning community, we prioritize working with individuals from a variety of backgrounds. We are continuously looking for talented professionals who share our values of collaboration, innovation, and commitment to excellence. If you are excited about joining a global community of AI builders, we encourage you to explore our careers page where you can find roles suited for your skill set.

## Explore
### Platforms and Services
- **Models:** Explore the diverse and powerful AI models created for a wide range of applications including NLP, image processing, and more.
- **Datasets:** Gain access to a vast library of datasets that span a wide cross-section of research domains.
- **Spaces:** Experience collaborative spaces where teams can build their own AI applications and share their progress.

### Community
Join our lively and dynamic community of AI builders. Share your work and learn from others with regular events, discussions, forums, and more. Let's build the future of AI together.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [23]:
def stream_brochure(company_name, url):
    MODEL = 'qwen2.5:3b'
    print(f"Streaming a brochure for {company_name} by calling {MODEL}")
    stream = ollama.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)


In [24]:
stream_brochure("HuggingFace", "https://huggingface.co")


Streaming a brochure for HuggingFace by calling qwen2.5:3b
Selecting relevant links for https://huggingface.co by calling qwen2.5:3b
{
    "links": [
        {"type": "about page", "url": "https://huggingface.co/"},
        {"type": "careers page", "url": "https://apply.workable.com/huggingface/"},
        {"type": "enterprise page", "url": "https://huggingface.co/enterprise"},
        {"type": "blog page", "url": "https://huggingface.co/blog"},
        {"type": "learn page", "url": "https://huggingface.co/learn"}
    ]
}
Found 5 relevant links


# HuggingFace Brochure

## About HuggingFace
At HuggingFace, we are a pioneer in the Machine Learning community, dedicated to fostering collaboration and innovation. Our platform serves as a hub where model developers and researchers can exchange models, datasets, and applications. With over 1M applications and 2M+ models, we are pivotal in enabling progress for the greater machine learning community.

## Our Products and Services
**Models**: Explore an extensive ecosystem of models, including those for various tasks like text to image, image to text, and more, with frequent updates.
**Datasets**: We curate over 500k datasets, serving as a rich resource for training and testing your models.
**Spaces**: Experience our feature-packed models and datasets in a custom environment with our innovative Spaces.
**Buckets**: With AI-native object storage, we offer robust and efficient solutions for managing data. 

## Our Culture
At HuggingFace, we thrive in an environment that values collaboration, innovation and open communication. Our company is renowned for its friendly and supportive teams, fostering an inclusive culture that welcomes a diverse range of perspectives and ideas.

## Our Team
We have opportunities for a variety of roles including Research Scientists, Software Engineers, and Customer Success Managers. Our team values creativity, collaboration, and learning, creating an environment that encourages growth and passion for our work. 

## Our Customers
Our diverse list of customers ranges from the tech giants like Google and Anthropic to researchers and small startups. Our commitment to excellence and commitment to fostering a supportive community means we cater to users at every level, providing tools that help them achieve their goals.

## Hugging Face Jobs
Stay connected for the latest openings. Explore the opportunities available at HuggingFace and help shape the future of Machine Learning with us.

## Contact Us
For enterprise support, further inquiry about pricing, or if you're interested in our team & enterprise plans, all inquiries are equally welcomed and thoroughly attended to.

---

At HuggingFace, we aim to be a beacon of innovation and community building. With the right team and cutting-edge technology, we are continuously pushing the boundaries of what’s possible. Join us in shaping a brighter future for the machine learning landscape.

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>